# Attention is All you Need 모델 - Transformer 모듈로 코드 수정

참고
> https://medium.com/data-science/build-your-own-transformer-from-scratch-using-pytorch-84c850470dcb

를 바탕으로 GPT와 구현함.

### Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math

import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### MultiHeadAttention & Position-wise Feed-Forward Networks

PyTorch의 `nn.Transformer` 또는 `nn.TransformerEncoderLayer`/`nn.TransformerDecoderLayer`를 사용할 경우: **MultiHeadAttention, Position-wise Feed-Forward Networks** 구현 필요 x   
1. `nn.Transformer`는 이미 내부에 모든 구성요소를 포함함
  - `MultiHeadAttention` = `nn.MultiheadAttention`
  - `PositionWiseFeedForward` = `nn.Linear -> ReLU -> nn.Linear`
  - `Residual` + `LayerNorm` = 이미 각 sublayer 안에 포함

2. `nn.TransformerEncoderLayer`, `nn.TransformerDecoderLayer`는 **Attention + FFN + LayerNorm** 구조를 기본으로 가짐
  - 사용자가 직접 분리 구현할 필요없이 클래스를 불러서 layer stacking만 하면 됨

3. 직접 구현은 학습 목적 또는 커스터마이징할 경우에만 필요함
  - 예) Attention 타입을 바꾸거나, 구조를 변경하려는 경우

### Positional Encoding

- 기존 코드와 거의 동일
- 차이점: `dropout` 추가
  - Transformer 모델에서 embedding + positional encoding 결과에 `dropout`을 적용하는 이유
    1. 과적합 방지
      - embedding + positional encoding은 모델의 첫 입력이자 모든 레이어를 거쳐 전달되는 정도의 출발점
      - 이 첫 입력을 통해 overfitting 발생 가능성
      - `dropout`으로 입력의 일부를 무작위로 제거해 일반화 성능 향상
    2. 다른 레이어들과의 일관성
      - Transformer의 모든 sub-layer ouput(`attention`, `feed-forward` 등)에는 `dropout` 적용됨
      - positional encoding에도 `dropout`을 걸어주는 건 그 구조와 학습 방식의 일관성 유지에 도움
- transformer에서 dropout 실행해줄 것으로 다시 제거함

In [4]:
class PositionalEncoding(nn.Module):
    # d_model: 임베딩 차원, max_deq_length: 최대 시퀀스 길이
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        # 전체 시퀀스 길이와 차원 수에 맞는 위치 벡터를 0으로 초기화
        pe = torch.zeros(max_seq_length, d_model)

        # position[0], [1], [2], ..., [max_seq_length-1] 형태로 위치 벡터 생성
        # .unsqueeze(1)은 2D로 변환 (모양: [max_seq_length, 1])
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)

        # sin, cos의 주기를 다르게 해주는 인자
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        # 짝수 인덱스는 sin, 홀수 인덱스는 cos을 사용
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # pe를 학습 파라미터는 아니지만 모델의 일부로 등록
        # 학습 중에는 업데이트 되지 않지만 GPU 메모리에 저장되어 forward 시 사용됨
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: (1, max_seq_len, d_model)

    def forward(self, x):
        # 입력 임베딩 x에 위치 인코딩을 더함
        # self.pe[:, :x.size(1)]는 입력 길이에 맞는 위치 벡터만 사용
        return x + self.pe[:, :x.size(1)] # x shape: (batch_size, seq_len, d_model)

### EncoderLayer & DecoderLayer

- `nn.TransformerEncoderLayer`, `nn.TransformerDecoderLayer`로 구현 가능
- EncoderLayer 포함 유소  
  - MultiHeadAttention, LayerNorm, Dropout, PositionWiseFeedForward
- DecoderLayer 포함 요소
  - Masked MultiHeadAttention, Cross-Attention, LayerNorm, Dropout, PositionWiseFeedForward


### TransformerModel

- PositionalEncoding의 dropout 문제
  - PositionalEncoding class: 위치 정보만 더해주는 역할
  - TransformerModel forward: dropout 추가해줌
    - `src = self.dropout(self.positional_encoding(self.encoder_embedding(src)))`
    - tgt = `self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))`
    


- generate_mask() 함수 삭제함
    - 직접 구현한 transformer 구조의 마스크 형태와 다른 형식 필요
    - PyTorch의 `nn.Transformer` 계열 모듈: 세 종류의 마스크를 분리해서 받음
    - 따라서 역할별 마스크를 따로 제공     

| 마스크 이름                 | Shape              | 역할                                                             |   
|----------------------------|--------------------|------------------------------------------------------------------|   
| `tgt_mask`                | (tgt_len, tgt_len) | 미래 차단 (no-peak mask), 디코더의 자기 회귀 형태 유지              |   
| `src_key_padding_mask`    | (batch, src_len)   | 소스 문장의 패딩 토큰 무시                                         |   
| `tgt_key_padding_mask`    | (batch, tgt_len)   | 타겟 문장의 패딩 토큰 무시                                         |   
| `memory_key_padding_mask` | (batch, src_len)   | 인코더 출력 마스킹 (일반적으로 `src_key_padding_mask`와 동일하게 사용) |   


In [6]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(TransformerModel, self).__init__()

        # 소스, 타겟 임베딩
        # 1. 소스 문장을 임베딩 벡터로 전환 (단어 ID -> d_model 차원 벡터)
        self.src_embedding = nn.Embedding(src_vocab_size, d_model) # == encoder_embedding
        # 2. 타켓 문장을 임베딩 벡터로 변환
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model) # == decoder_embedding
        # 3. 위치 정보를 임베딩에 추가
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        # 인코더 레이어와 전체 인코더
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='relu', # PositionWiseFeedForward 내부 활성화 함수 지정
            batch_first=True # 입력 텐서의 shape 순서를 (batch, seq_len, d_model)로 고정
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 디코더 레이어와 전체 디코더
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='relu', # PositionWiseFeedForward 내부 활성화 함수 지정
            batch_first=True # 입력 텐서의 shape 순서를 (batch, seq_len, d_model)로 고정
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # 최종 출력 → 단어 분포
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        # 드롭아웃
        self.dropout = nn.Dropout(dropout)


    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None, tgt_mask=None):
        # 1. 임베딩 + 위치 인코딩
        src_emb = self.dropout(self.positional_encoding(self.src_embedding(src)))  # (batch, seq, d_model)
        tgt_emb = self.dropout(self.positional_encoding(self.tgt_embedding(tgt)))

        # 2. 인코더
        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

        # 3. 디코더
        output = self.decoder(tgt_emb, memory,
                              tgt_mask=tgt_mask,
                              tgt_key_padding_mask=tgt_key_padding_mask,
                              memory_key_padding_mask=memory_key_padding_mask)

        # 4. 최종 출력층
        return self.fc_out(output)  # (batch, tgt_seq_len, vocab_size)

### Preparing Sample Data

In [7]:
# 1. 하이퍼파라미터 설정
src_vocab_size = 5000        # 입력 단어 사전 크기
tgt_vocab_size = 5000        # 출력 단어 사전 크기
d_model = 512                # 임베딩 및 모델 차원
num_heads = 8                # 멀티 헤드 수
num_layers = 6               # 인코더/디코더 레이어 수
d_ff = 2048                  # FeedForward 내부 차원
max_seq_length = 100         # 최대 시퀀스 길이
dropout = 0.1                # 드롭아웃 비율

# 2. Transformer 모델 초기화
model = TransformerModel(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_seq_length=max_seq_length,
    dropout=dropout
)

# 3. 샘플 입력 데이터 생성
batch_size = 64
src_data = torch.randint(1, src_vocab_size, (batch_size, max_seq_length))  # (batch, seq_len)
tgt_data = torch.randint(1, tgt_vocab_size, (batch_size, max_seq_length))  # (batch, seq_len)

### Training the model

In [8]:
# 손실 함수 정의
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 옵티마이저 정의
optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

# 모델 학습 모드로 전환
model.train()

# No-Peak 마스크 생성 함수 (미래 정보 차단용)
def generate_square_subsequent_mask(seq_len):
    return torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)

# 총 10 에포크 동안 학습
for epoch in range(10):
    optimizer.zero_grad()

    # 1. 입력 마스크 생성
    tgt_seq_len = tgt_data.size(1)
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len).to(tgt_data.device)  # (tgt_seq_len, tgt_seq_len)

    src_key_padding_mask = (src_data == 0)  # (batch, src_seq_len)
    tgt_key_padding_mask = (tgt_data[:, :-1] == 0)  # (batch, tgt_seq_len - 1)

    # 2. 디코더 입력은 <sos> A B 형태, 정답은 A B <eos>
    decoder_input = tgt_data[:, :-1]
    target_output = tgt_data[:, 1:]

    # 3. 모델 실행
    output = model(
        src=src_data,
        tgt=decoder_input,
        src_key_padding_mask=src_key_padding_mask,
        tgt_key_padding_mask=tgt_key_padding_mask,
        memory_key_padding_mask=src_key_padding_mask,
        tgt_mask=tgt_mask[:decoder_input.size(1), :decoder_input.size(1)]
    )

    # 4. 손실 계산
    loss = criterion(output.view(-1, tgt_vocab_size), target_output.reshape(-1))

    # 5. 역전파 및 가중치 업데이트
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch 1, Loss: 8.6815
Epoch 2, Loss: 8.5835
Epoch 3, Loss: 8.5297
Epoch 4, Loss: 8.4931
Epoch 5, Loss: 8.4624
Epoch 6, Loss: 8.4288
Epoch 7, Loss: 8.3961
Epoch 8, Loss: 8.3662
Epoch 9, Loss: 8.3197
Epoch 10, Loss: 8.2753
